# Step 4: RQ1 Schema-Level Fidelity Check

Before evaluating RQ1 LLM performance, we first assess schema-level fidelity by checking whether each returned feature belongs to the supplied feature list, whether duplicate features appear, whether the required number of ranked features is returned, and whether the response follows the required JSON structure. This check ensures that the output satisfies the constrained format before Top-K overlap and Kendall's tau are used to evaluate ranking preservation.

This notebook is intentionally separate from the existing RQ1 notebooks. It reads completed RQ1 result folders and writes schema-fidelity outputs alongside each model's existing result directory without modifying the original notebooks or metric files.

## 1. Setup

Define project paths and the readable/raw feature name mapping used by the RQ1 parsers.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd()
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_ClubLending")


def find_local_project_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Data" / "ModifiedData").exists() and (candidate / "Result").exists():
            return candidate
    return start.parent if start.name == "Code" else start


BASE_DIR = COLAB_PROJECT_ROOT if COLAB_PROJECT_ROOT.exists() else find_local_project_root(PROJECT_ROOT)
RESULT_ROOT = BASE_DIR / "Result"
RESULT_PROJECT_DIR = RESULT_ROOT / "LendingClub" if (RESULT_ROOT / "LendingClub").exists() else RESULT_ROOT
SHAP_DIR = RESULT_PROJECT_DIR / "SHAP"
SHAP_RETRAINED_DIR = RESULT_PROJECT_DIR / "SHAP_retrained"
LLM_DIR = RESULT_PROJECT_DIR / "LLM"

print(f"Base directory: {BASE_DIR}")
print(f"Result directory: {RESULT_PROJECT_DIR}")
print(f"LLM directory: {LLM_DIR}")

FEATURE_TO_NAME_MAP = {
    "home_ownership": "Home Ownership Status",
    "verification_status": "Income Verification Status",
    "purpose": "Loan Purpose",
    "area": "Borrower Area",
    "addr_state": "Borrower State",
    "term": "Number of Payments",
    "grade": "Loan Grade",
    "sub_grade": "Loan Sub Grade",
    "emp_length": "Employment Length",
    "pub_rec_bankruptcies": "Number of Public Bankruptcies",
    "funded_amnt": "Loan Amount",
    "int_rate": "Interest Rate",
    "installment": "Monthly Payment",
    "annual_inc": "Annual Income",
    "dti": "Debt to Income Ratio",
    "delinq_2yrs": "Number of Delinquencies in Past 2 Years",
    "fico_range_low": "Lowest FICO Score",
    "fico_range_high": "Highest FICO Score",
    "inq_last_6mths": "Number of Inquiries in Last 6 Months",
    "open_acc": "Number of Open Accounts",
    "pub_rec": "Number of Derogatory Public Records",
    "revol_bal": "Revolving Balance",
    "revol_util": "Revolving Utilization Rate",
    "total_acc": "Total Number of Accounts",
}
NAME_TO_FEATURE_MAP = {v: k for k, v in FEATURE_TO_NAME_MAP.items()}


## 2. Experiment Inputs

The check uses the same completed RQ1 result folders. Logistic uses `Logistic_Contrib.xlsx`; XGBoost uses `XGBoost_SHAP_Aggregated.xlsx`.

In [ ]:
RQ1_SPECS = {
    "Logistic": {
        "result_dir": LLM_DIR / "RQ1_Logistic_Redesigned",
        "sample_path": LLM_DIR / "500_samples_logistic.xlsx",
        "attribution_path": SHAP_DIR / "Logistic_Contrib.xlsx",
        "attribution_type": "wide_abs",  # rows are samples; columns are original features
    },
    "XGBoost": {
        "result_dir": LLM_DIR / "RQ1_XGBoost_Redesigned",
        "sample_path": LLM_DIR / "500_samples_xgboost_matched.xlsx",
        "attribution_path": SHAP_RETRAINED_DIR / "XGBoost_SHAP_Aggregated.xlsx",
        "attribution_type": "long_abs",  # one row per sample-feature pair
    },
}

for family, spec in RQ1_SPECS.items():
    print()
    print(f"{family}")
    for key, value in spec.items():
        print(f"  {key}: {value}")


## 3. Helper Functions

These functions reconstruct the supplied feature list for each sample, load the parsed LLM rankings, and evaluate schema-level fidelity. The strict JSON-structure check is run only when raw successful responses are available; existing `batch_*.xlsx` files contain parsed output rather than raw JSON.

In [ ]:
def normalize_feature_text(text):
    text = str(text).strip()
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()


def feature_aliases(feature):
    aliases = {str(feature)}
    if feature in FEATURE_TO_NAME_MAP:
        aliases.add(FEATURE_TO_NAME_MAP[feature])
    if feature in NAME_TO_FEATURE_MAP:
        aliases.add(NAME_TO_FEATURE_MAP[feature])
    return {alias for alias in aliases if alias is not None and str(alias).strip()}


def build_allowed_feature_lookup(allowed_features):
    lookup = {}
    for feature in allowed_features:
        for alias in feature_aliases(feature):
            lookup[normalize_feature_text(alias)] = feature
    return lookup


def canonicalize_feature(feature, allowed_features):
    lookup = build_allowed_feature_lookup(allowed_features)
    normalized = normalize_feature_text(feature)
    if normalized in lookup:
        return lookup[normalized]

    # Conservative fallback for small wording variations in readable names.
    for alias_key, allowed_feature in lookup.items():
        if alias_key and alias_key in normalized:
            return allowed_feature
    return None


def load_sample_df(spec):
    sample_df = pd.read_excel(spec["sample_path"])
    sample_df["Rowindex"] = sample_df["Rowindex"].astype(int)
    return sample_df


def load_attribution(spec):
    if spec["attribution_type"] == "wide_abs":
        df = pd.read_excel(spec["attribution_path"], index_col=0)
        df.index = df.index.astype(int)
        return df

    df = pd.read_excel(spec["attribution_path"])
    df["Rowindex"] = df["Rowindex"].astype(int)
    if "AbsSHAPValue" not in df.columns:
        if "AbsContribution" in df.columns:
            df["AbsSHAPValue"] = df["AbsContribution"].abs()
        elif "SignedContribution" in df.columns:
            df["AbsSHAPValue"] = df["SignedContribution"].abs()
        else:
            raise KeyError("XGBoost attribution table needs AbsSHAPValue, AbsContribution, or SignedContribution.")
    return df


def reference_rank_for_row(rowindex, attribution_df, attribution_type):
    rowindex = int(rowindex)
    if attribution_type == "wide_abs":
        row = attribution_df.loc[rowindex]
        return row.abs().sort_values(ascending=False).index.tolist()

    row = attribution_df[attribution_df["Rowindex"] == rowindex].copy()
    if row.empty:
        raise KeyError(f"Rowindex {rowindex} is missing from attribution table.")
    return row.sort_values("AbsSHAPValue", ascending=False)["Feature"].astype(str).tolist()


def model_output_dirs(result_dir):
    if not result_dir.exists():
        return []
    return sorted([p for p in result_dir.iterdir() if p.is_dir() and not p.name.startswith(".")])


def load_llm_batches(model_dir):
    batch_files = sorted(model_dir.glob("batch_*.xlsx"), key=lambda p: int(re.search(r"batch_(\d+)", p.stem).group(1)))
    frames = []
    for path in batch_files:
        frame = pd.read_excel(path)
        frame["SourceFile"] = path.name
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=["Rowindex", "FeatureRank", "Feature", "SourceFile"])
    df = pd.concat(frames, ignore_index=True)
    df["Rowindex"] = df["Rowindex"].astype(int)
    df["FeatureRank"] = pd.to_numeric(df["FeatureRank"], errors="coerce")
    return df


def load_error_log(model_dir):
    path = model_dir / "errors.jsonl"
    if not path.exists():
        return pd.DataFrame(columns=["Rowindex", "Error"])
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                rows.append({"RawLine": line})
    if not rows:
        return pd.DataFrame(columns=["Rowindex", "Error"])
    df = pd.DataFrame(rows)
    if "Rowindex" in df.columns:
        df["Rowindex"] = pd.to_numeric(df["Rowindex"], errors="coerce").astype("Int64")
    return df


def find_raw_response_file(model_dir):
    candidates = [
        "raw_responses.xlsx",
        "responses.xlsx",
        "raw_responses.jsonl",
        "responses.jsonl",
    ]
    for name in candidates:
        path = model_dir / name
        if path.exists():
            return path
    return None


def load_raw_responses(model_dir):
    path = find_raw_response_file(model_dir)
    if path is None:
        return pd.DataFrame(columns=["Rowindex", "RawResponse"]), None

    if path.suffix == ".xlsx":
        df = pd.read_excel(path)
    else:
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
        df = pd.DataFrame(rows)

    row_col = "Rowindex" if "Rowindex" in df.columns else "sample_index" if "sample_index" in df.columns else None
    raw_col = "RawResponse" if "RawResponse" in df.columns else "response" if "response" in df.columns else None
    if row_col is None or raw_col is None:
        raise KeyError(f"Raw response file {path} must contain Rowindex/sample_index and RawResponse/response columns.")

    out = df[[row_col, raw_col]].rename(columns={row_col: "Rowindex", raw_col: "RawResponse"})
    out["Rowindex"] = out["Rowindex"].astype(int)
    return out, path


def strict_json_structure_pass(raw_response):
    try:
        parsed = json.loads(str(raw_response))
    except Exception:
        return False
    if not isinstance(parsed, dict):
        return False
    ranked = parsed.get("ranked_features")
    if not isinstance(ranked, list):
        return False
    return all(isinstance(item, (str, dict)) for item in ranked)


## 4. Schema-Level Fidelity Evaluation

Each sample receives one row with the following checks:

- `UnknownFeaturePass`: every returned feature maps to the supplied feature list.
- `DuplicateFeaturePass`: no repeated canonical features are returned.
- `RequiredCountPass`: the number of returned ranked features equals the number of supplied features.
- `JSONStructurePass`: strict JSON validity when raw successful responses are available; otherwise unavailable.
- `SchemaLevelPass`: membership, duplicate, count, and parse availability checks.
- `StrictSchemaLevelPass`: same as `SchemaLevelPass` plus strict JSON structure, only available when raw responses exist.

In [ ]:
def evaluate_schema_for_model(family, spec, model_dir):
    sample_df = load_sample_df(spec)
    attribution_df = load_attribution(spec)
    parsed_df = load_llm_batches(model_dir)
    error_df = load_error_log(model_dir)
    raw_df, raw_path = load_raw_responses(model_dir)

    raw_by_row = {}
    if not raw_df.empty:
        raw_by_row = raw_df.drop_duplicates("Rowindex").set_index("Rowindex")["RawResponse"].to_dict()

    rows = []
    for rowindex in sample_df["Rowindex"].astype(int):
        allowed_features = reference_rank_for_row(rowindex, attribution_df, spec["attribution_type"])
        allowed_set = set(allowed_features)
        model_rows = parsed_df[parsed_df["Rowindex"] == rowindex].sort_values("FeatureRank")

        returned_raw = model_rows["Feature"].astype(str).tolist() if not model_rows.empty else []
        canonical = []
        unknown = []
        for feature in returned_raw:
            mapped = canonicalize_feature(feature, allowed_features)
            if mapped is None:
                unknown.append(feature)
            else:
                canonical.append(mapped)

        duplicate_features = sorted({feature for feature in canonical if canonical.count(feature) > 1})
        returned_count = len(returned_raw)
        required_count = len(allowed_features)
        has_parsed_output = returned_count > 0
        unknown_pass = len(unknown) == 0
        duplicate_pass = len(duplicate_features) == 0
        count_pass = returned_count == required_count

        raw_response = raw_by_row.get(rowindex)
        json_pass = np.nan if raw_response is None else strict_json_structure_pass(raw_response)

        schema_level_pass = bool(has_parsed_output and unknown_pass and duplicate_pass and count_pass)
        strict_schema_pass = np.nan if pd.isna(json_pass) else bool(schema_level_pass and json_pass)

        rows.append({
            "Family": family,
            "Model": model_dir.name,
            "Rowindex": rowindex,
            "HasParsedOutput": has_parsed_output,
            "ReturnedCount": returned_count,
            "RequiredCount": required_count,
            "RequiredCountPass": count_pass,
            "UnknownFeatureCount": len(unknown),
            "UnknownFeaturePass": unknown_pass,
            "UnknownFeatures": "; ".join(map(str, unknown)),
            "DuplicateFeatureCount": len(duplicate_features),
            "DuplicateFeaturePass": duplicate_pass,
            "DuplicateFeatures": "; ".join(map(str, duplicate_features)),
            "SchemaLevelPass": schema_level_pass,
            "JSONStructurePass": json_pass,
            "StrictSchemaLevelPass": strict_schema_pass,
            "RawResponseFile": str(raw_path) if raw_path is not None else "",
        })

    result = pd.DataFrame(rows)

    if not error_df.empty and "Rowindex" in error_df.columns:
        error_counts = error_df.dropna(subset=["Rowindex"]).groupby("Rowindex").size().rename("ErrorLogCount")
        result = result.merge(error_counts, how="left", left_on="Rowindex", right_index=True)
        result["ErrorLogCount"] = result["ErrorLogCount"].fillna(0).astype(int)
    else:
        result["ErrorLogCount"] = 0

    return result


def summarize_schema_fidelity(schema_df):
    if schema_df.empty:
        return pd.DataFrame()
    rows = []
    group_cols = ["Family", "Model"]
    for (family, model), group in schema_df.groupby(group_cols):
        row = {
            "Family": family,
            "Model": model,
            "N": len(group),
            "ParsedOutputRate": group["HasParsedOutput"].mean(),
            "UnknownFeaturePassRate": group["UnknownFeaturePass"].mean(),
            "DuplicateFeaturePassRate": group["DuplicateFeaturePass"].mean(),
            "RequiredCountPassRate": group["RequiredCountPass"].mean(),
            "SchemaLevelPassRate": group["SchemaLevelPass"].mean(),
            "MeanUnknownFeatureCount": group["UnknownFeatureCount"].mean(),
            "MeanDuplicateFeatureCount": group["DuplicateFeatureCount"].mean(),
            "MeanReturnedCount": group["ReturnedCount"].mean(),
            "MeanRequiredCount": group["RequiredCount"].mean(),
            "TotalErrorLogCount": group["ErrorLogCount"].sum(),
        }
        if group["JSONStructurePass"].notna().any():
            row["JSONStructurePassRate"] = group["JSONStructurePass"].dropna().mean()
            row["StrictSchemaLevelPassRate"] = group["StrictSchemaLevelPass"].dropna().mean()
        else:
            row["JSONStructurePassRate"] = np.nan
            row["StrictSchemaLevelPassRate"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["Family", "Model"]).reset_index(drop=True)


## 5. Run All Completed RQ1 Models

This cell scans both redesigned RQ1 folders and saves two files for each model directory:

- `schema_fidelity_by_sample.xlsx`
- `schema_fidelity_summary.xlsx`

It also writes one combined project-level summary under `Result/LLM/RQ1_Schema_Fidelity_Check/`.

In [ ]:
all_schema_frames = []
all_summary_frames = []

for family, spec in RQ1_SPECS.items():
    print(f"\n=== {family} ===")
    model_dirs = model_output_dirs(spec["result_dir"])
    if not model_dirs:
        print(f"No model result directories found under {spec['result_dir']}")
        continue

    for model_dir in model_dirs:
        print(f"Checking {model_dir.name}")
        schema_df = evaluate_schema_for_model(family, spec, model_dir)
        summary_df = summarize_schema_fidelity(schema_df)

        sample_out = model_dir / "schema_fidelity_by_sample.xlsx"
        summary_out = model_dir / "schema_fidelity_summary.xlsx"
        schema_df.to_excel(sample_out, index=False)
        summary_df.to_excel(summary_out, index=False)

        print(f"  saved: {sample_out}")
        print(f"  saved: {summary_out}")
        display(summary_df)

        all_schema_frames.append(schema_df)
        all_summary_frames.append(summary_df)

combined_schema_df = pd.concat(all_schema_frames, ignore_index=True) if all_schema_frames else pd.DataFrame()
combined_summary_df = pd.concat(all_summary_frames, ignore_index=True) if all_summary_frames else pd.DataFrame()

SCHEMA_OUT_DIR = LLM_DIR / "RQ1_Schema_Fidelity_Check"
SCHEMA_OUT_DIR.mkdir(parents=True, exist_ok=True)

combined_schema_path = SCHEMA_OUT_DIR / "schema_fidelity_by_sample_all_models.xlsx"
combined_summary_path = SCHEMA_OUT_DIR / "schema_fidelity_summary_all_models.xlsx"
combined_schema_df.to_excel(combined_schema_path, index=False)
combined_summary_df.to_excel(combined_summary_path, index=False)

print(f"\nSaved combined sample-level schema file: {combined_schema_path}")
print(f"Saved combined schema summary file: {combined_summary_path}")
display(combined_summary_df)


## 6. Inspect Failures

Use this section to inspect rows that fail one or more schema checks before interpreting ranking metrics.

In [ ]:
if combined_schema_df.empty:
    print("Run the previous cell first.")
else:
    failed = combined_schema_df[
        (~combined_schema_df["SchemaLevelPass"]) |
        (combined_schema_df["JSONStructurePass"].notna() & ~combined_schema_df["JSONStructurePass"])
    ].copy()

    print(f"Failed or incomplete schema rows: {len(failed)}")
    display(
        failed[[
            "Family", "Model", "Rowindex", "ReturnedCount", "RequiredCount",
            "UnknownFeatureCount", "UnknownFeatures",
            "DuplicateFeatureCount", "DuplicateFeatures",
            "RequiredCountPass", "UnknownFeaturePass", "DuplicateFeaturePass",
            "JSONStructurePass", "SchemaLevelPass", "ErrorLogCount",
        ]].head(100)
    )


## 7. Manuscript-Ready Table

This creates a compact table that can be reported before Overlap@K and Kendall's tau.

In [ ]:
if combined_summary_df.empty:
    print("Run the evaluation cell first.")
else:
    report_cols = [
        "Family", "Model", "N",
        "ParsedOutputRate",
        "UnknownFeaturePassRate",
        "DuplicateFeaturePassRate",
        "RequiredCountPassRate",
        "SchemaLevelPassRate",
        "JSONStructurePassRate",
        "TotalErrorLogCount",
    ]
    report_table = combined_summary_df[report_cols].copy()
    for col in report_table.columns:
        if col.endswith("Rate"):
            report_table[col] = report_table[col].round(4)
    display(report_table)

    report_path = SCHEMA_OUT_DIR / "schema_fidelity_report_table.xlsx"
    report_table.to_excel(report_path, index=False)
    print(f"Saved manuscript-ready table: {report_path}")
